In [6]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

In [8]:
from google.auth.transport.requests import Request
from google.oauth2.service_account import Credentials
from googleapiclient.discovery import build

# Authentification avec un fichier de clé JSON (service account)
# Ou utiliser OAuth2 pour l'authentification utilisateur

# Option 1 : Avec un fichier credentials.json (OAuth2)
from google_auth_oauthlib.flow import InstalledAppFlow

SCOPES = ['https://www.googleapis.com/auth/drive']

flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)
creds = flow.run_local_server(port=0)

service = build('drive', 'v3', credentials=creds)

# Lister les fichiers
results = service.files().list(spaces='drive', pageSize=10).execute()
files = results.get('files', [])
print(files)

FileNotFoundError: [Errno 2] No such file or directory: 'credentials.json'

In [2]:
# Authentification Google Drive OAuth2 (une seule fois)
import os
import pickle
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
import h5py
import pandas as pd
from io import BytesIO

SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
TOKEN_FILE = 'token.pickle'
CREDENTIALS_FILE = 'credentials.json'

def authenticate():
    """Authentiffication Google Drive (une seule fois)"""
    creds = None
    
    # Charger le token sauvegardé s'il existe
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE, 'rb') as token:
            creds = pickle.load(token)
        print("✅ Token chargé (authentification précédente)\n")
    
    # Si pas de token valide, créer une nouvelle authentification
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
            print("🔄 Token rafraîchi\n")
        else:
            # Première authentification
            if not os.path.exists(CREDENTIALS_FILE):
                print("❌ ERREUR : Fichier 'credentials.json' non trouvé !")
                print("\n📥 Pour obtenir credentials.json :")
                print("   1. Allez sur https://console.cloud.google.com/")
                print("   2. Créez un OAuth 2.0 Client ID (Application de bureau)")
                print("   3. Téléchargez le fichier JSON")
                print("   4. Placez-le à la racine du workspace en tant que 'credentials.json'")
                print("\nAprès, réexécutez cette cellule !")
                return None
            
            print("🔐 Première authentification Google Drive...")
            print("   Une page s'ouvrira pour confirmer l'accès\n")
            
            flow = InstalledAppFlow.from_client_secrets_file(CREDENTIALS_FILE, SCOPES)
            creds = flow.run_local_server(port=0)
        
        # Sauvegarder le token pour les futures utilisations
        with open(TOKEN_FILE, 'wb') as token:
            pickle.dump(creds, token)
    
    return creds

# Authentification
creds = authenticate()

if creds:
    print("✅ Connecté à Google Drive !\n")
    
    # Créer le service Google Drive API
    service = build('drive', 'v3', credentials=creds)
    
    # ID du fichier Aircraft_01.h5
    FILE_ID = "1klSTn-_kQFVoDCvH7rwbFLAteXroAj02"
    
    print(f"📥 Lecture du fichier Aircraft_01.h5 depuis Google Drive...")
    
    try:
        # Obtenir les métadonnées du fichier
        file_metadata = service.files().get(fileId=FILE_ID, fields='name, size').execute()
        print(f"   Nom : {file_metadata['name']}")
        print(f"   Taille : {int(file_metadata.get('size', 0)) / (1024*1024):.1f} MB\n")
        
        # Télécharger le fichier en mémoire
        request = service.files().get_media(fileId=FILE_ID)
        file_buffer = BytesIO()
        downloader = MediaIoBaseDownload(file_buffer, request)
        
        done = False
        while not done:
            status, done = downloader.next_chunk()
            if status:
                print(f"   Téléchargement : {int(status.progress() * 100)}%")
        
        file_buffer.seek(0)
        print("\n✅ Fichier chargé en mémoire !\n")
        
        # Ouvrir le fichier HDF5
        with h5py.File(file_buffer, 'r') as f:
            records = list(f.keys())
            print(f"📊 Fichier accessible !")
            print(f"   Records : {records}\n")
            
            # Afficher infos sur chaque record
            for record in records[:3]:
                print(f"   📋 Record '{record}' :")
                print(f"      - Capteurs : {len(f[record].keys())}")
                print(f"      - Premiers : {list(f[record].keys())[:3]}...\n")
        
        print("✅ Prêt à manipuler les données !")
        
    except Exception as e:
        print(f"❌ Erreur : {e}")

🔐 Première authentification Google Drive...
   Une page s'ouvrira pour confirmer l'accès

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=271620978937-da5hk52tlh743qg5h5mq2d86pc288jor.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A48969%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive.readonly&state=4w2Hpj4ua2EKCQGIsmtVP30L979O3h&code_challenge=a7o468IhNbOEriD6H0-SnJYlaPo04G7pp6ejnozHvDs&code_challenge_method=S256&access_type=offline


KeyboardInterrupt: 